# ASR evaluation: Whisper large-v3 vs large-v3-turbo on our own recordings of team30

**What this notebook does.** Diagnostic, TODO §13 Phase 2. It measures the speech → English
stage on the team's 30 sentences, spoken by us:

1. **Record** each sentence once (interactive, 16 kHz mono). Existing files are skipped.
2. **Transcribe** with every model in `asr.compare_models` (config): WER and latency.
3. **End to end:** gloss the ASR text vs gloss the true English. The gap is how much ASR
   errors cost downstream.

`large-v3` is the team's model; `large-v3-turbo` is what Workers AI hosts (§12.5).

| artifact | path |
|---|---|
| recordings | `data/raw/speech30/<id>.wav` |
| transcripts | `data/cache/synthesis/asr/<model>.json` (one per model, per-sentence) |
| WER / latency table | `data/cache/synthesis/asr/asr_scores.csv` |
| end-to-end gloss scores | `data/cache/synthesis/asr/e2e_scores.csv` |

**Resumable:** recording skips existing wavs, and transcription skips sentences already
in a model's JSON (written after each one).

**Design decisions vs the TODO spec:** Google Speech Commands is not used (keyword
spotting, no sentences; report §7.1). Your own recordings are the sentence-level speech
with transcripts that the plan called for. `best_of` is dropped (inert at temperature 0),
and peak normalization is applied to the audio Whisper actually gets (the team
notebook's was dead code).

## Setup

In [ ]:
# ============================================================
# Imports, configs, paths
# ============================================================
import json
import os

import pandas as pd
import soundfile as sf

from sb.core.paths import CACHE_DIR, EXPERIMENTS_DIR, ROOT_DIR
from sb.synthesize import asr, evalsets, metrics
from sb.synthesize.gloss import GlossEngine

CONFIG_DIR = EXPERIMENTS_DIR / "synthesis" / "configs"
SPEECH = json.loads((CONFIG_DIR / "speech.pipeline.json").read_text(encoding="utf-8"))
T2G = json.loads((CONFIG_DIR / "aslg.text2gloss.json").read_text(encoding="utf-8"))
REC_DIR = ROOT_DIR / SPEECH["recording"]["dir"]
OUT = CACHE_DIR / "synthesis" / "asr"
REC_DIR.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
T30 = evalsets.load("team30", T2G["eval"]["team30_version"])

print(f"recordings : {REC_DIR} ({len(list(REC_DIR.glob('*.wav')))}/{len(T30)} present)")
print(f"results    : {OUT}")
print(f"models     : {SPEECH['asr']['compare_models']}")

## 1. Record the 30 sentences

Set `RECORD = True`, run the cell, and read each sentence when it prints. Each recording
lasts `SECONDS`. Re-run to fill gaps; `REDO` lists ids to re-record.

In [ ]:
# ============================================================
# Interactive recorder (skips existing wavs)
# ============================================================
RECORD = False   # True -> record every missing sentence
SECONDS = SPEECH["recording"]["seconds"]
REDO = []        # e.g. [7, 19] -> record these again

if RECORD:
    import time

    for _, r in T30.iterrows():
        path = REC_DIR / f"{r.id:02d}.wav"
        if path.exists() and r.id not in REDO:
            continue
        print(f"\n[{r.id}/30] in 2 s, read aloud:  {r.english}")
        time.sleep(2)
        audio = asr.record(SECONDS)
        tmp = path.with_suffix(".tmp.wav")
        sf.write(tmp, audio, asr.SAMPLE_RATE)
        os.replace(tmp, path)
        print(f"   saved {path.name} (peak {abs(audio).max():.2f})")
print(f"{len(list(REC_DIR.glob('??.wav')))}/{len(T30)} recordings present")

## 2. Transcribe with each model

Each model's per-sentence transcripts go to `<model>.json` as they are made. WER uses the
team's normalization (lowercase, no punctuation). Numbers are compared as written, so
"3 o'clock" vs "three o'clock" counts as an error: that is a real downstream difference,
since the gloss engines see the digit.

In [ ]:
# ============================================================
# Transcribe (resumable per model) + WER / latency
# ============================================================
from tqdm.auto import tqdm

wavs = {int(p.stem): p for p in sorted(REC_DIR.glob("??.wav"))}
rows = []
for model in SPEECH["asr"]["compare_models"]:
    path = OUT / f"{model}.json"
    done = json.loads(path.read_text()) if path.exists() else {}
    todo = [i for i in wavs if str(i) not in done]
    if todo:
        tr = asr.Transcriber.from_config(SPEECH["asr"], model=model)
        for i in tqdm(todo, desc=f"ASR {model}"):
            t = tr(wavs[i])
            done[str(i)] = {"text": t.text, "seconds": t.seconds, "audio_seconds": t.audio_seconds}
            tmp = path.with_suffix(".tmp")
            tmp.write_text(json.dumps(done, indent=1))
            os.replace(tmp, path)
        del tr
    ids = sorted(int(k) for k in done)
    refs = [T30.english[i - 1] for i in ids]
    hyps = [done[str(i)]["text"] for i in ids]
    rows.append({"model": model, "n": len(ids), "wer": metrics.asr_wer(refs, hyps),
                 "sent_exact": sum(metrics.normalize_text(a) == metrics.normalize_text(b) for a, b in zip(refs, hyps)) / max(len(ids), 1),
                 "latency_median_s": pd.Series([done[str(i)]["seconds"] for i in ids]).median()})
tab = pd.DataFrame(rows).set_index("model")
tab.to_csv(OUT / "asr_scores.csv")
tab.round(3)

## 3. End to end: does ASR error survive into the gloss?

For every engine, gloss (a) the true English and (b) each model's transcript, and score
both against the draft references. The difference is the ASR cost. It is often smaller
than the WER, because many ASR errors are in words the gloss drops anyway (articles,
auxiliaries).

In [ ]:
# ============================================================
# Gloss of ASR text vs gloss of true text
# ============================================================
refs_all = {i + 1: metrics.normalize_gloss(g) for i, g in enumerate(T30.reference_draft)}
rows = []
for model in SPEECH["asr"]["compare_models"]:
    path = OUT / f"{model}.json"
    if not path.exists():
        continue
    done = json.loads(path.read_text())
    ids = sorted(int(k) for k in done)
    for name in T2G["eval"]["engines"]:
        eng = GlossEngine(name, T2G["t5"])
        refs = [refs_all[i] for i in ids]
        true = [metrics.normalize_gloss(r["gloss"]) for r in eng.gloss_sentences([T30.english[i - 1] for i in ids])]
        heard = [metrics.normalize_gloss(eng(done[str(i)]["text"]).gloss) for i in ids]
        a, b = metrics.corpus_scores(refs, true, meteor=False), metrics.corpus_scores(refs, heard, meteor=False)
        rows.append({"asr": model, "engine": name, "gloss_wer_true_text": a["gloss_wer"],
                     "gloss_wer_asr_text": b["gloss_wer"], "asr_cost": b["gloss_wer"] - a["gloss_wer"],
                     "bleu4_true_text": a["bleu4"], "bleu4_asr_text": b["bleu4"]})
e2e = pd.DataFrame(rows)
e2e.to_csv(OUT / "e2e_scores.csv", index=False)
e2e.round(3) if len(e2e) else "no transcripts yet -- run §1 and §2"